# cancer-recog-apoptosis — RUNG 9: does IFN rescue the blocker? (HLA-A vs the interferon program)

RUNG-8 found the Tmod blocker's antigen (HLA-A) is **low at rest in immune-privileged vital tissue** (heart
conduction, cardiomyocytes, neurons) → the blocker fails there. **But CAR-T therapy floods tissue with
IFN-γ, which upregulates MHC-I.** This run asks the one question that decides whether RUNG-8's safety hole is
real in vivo: **when the interferon program is ON in those cells, is HLA-A turned back on (blocker rescued)?**

Per vital cell type it compares HLA-A-low between **IFN-low** and **IFN-high** cells — with a **housekeeping-gene
depth control** so an apparent rescue that's really just deeper sequencing is flagged, not believed.

Same engineering as RUNG-8: **resumable** Drive tiles (re-run Cell 5 after any disconnect), **foreground
heartbeat**, **CPU runtime** (a ~17-gene fetch is IO-bound — no GPU needed).

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Google Drive for a RESUMABLE cache + start the run log
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon/rung9_tiles'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['RUNG9_CACHE'] = cache_dir
    print('[CELL 2] Drive mounted — RUNG9_CACHE =', cache_dir, '(tiles persist; re-run Cell 5 to resume)')
except Exception as e:
    os.environ['RUNG9_CACHE'] = '/content/cancer-recog-apoptosis/runs/rung9_ifn/tiles'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — tiles in /content (LOST on disconnect!)')
from scripts.runlog import new_log
RUNLOG = new_log('rung9_ifn', repo_dir='.')

In [ ]:
#@title Cell 3 — VALIDATE the math on synthetic ground truth (CPU, no data, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable, '-u', 'scripts/30_hla_ifn_inducibility.py', 'selftest'], RUNLOG)
print('[CELL 3]', '✓ validated (incl. the depth-confound flag)' if rc == 0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install CELLxGENE Census (only needed for the real run)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART, do it, then Run all — the Drive cache makes resume instant)')

In [ ]:
#@title Cell 4b — GPU check (transparent: this rung does NOT need a GPU)
# RUNG-9 pulls ~17 genes (HLA + ISG + housekeeping). Bottleneck is the Census FETCH (network/disk); the
# per-cell scoring is a trivial numpy op. Use a CPU runtime and save your GPU quota.
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    print('[GPU]', out or '(none)', '— NOT used this rung (fetch-bound).')
except Exception as e:
    print('[GPU] none present (', type(e).__name__, ') — fine, RUNG-9 is CPU/fetch-bound.')

In [ ]:
#@title Cell 5 — REAL run: HLA-A vs IFN program in vital tissue (RESUMABLE + heartbeat-logged)
#@markdown ~17-gene fetch. A background **[heartbeat]** prints every ~20s so you always see the step + RAM.
#@markdown Watch the **[N/9]** tissue progress. **If the cap disconnects you, just RE-RUN THIS CELL** — Drive
#@markdown tiles are skipped and it continues (re-aggregation is network-free).
import sys, os
from scripts.runlog import run_logged
print('[CELL 5] starting — watch [+s][rung9] step lines and [heartbeat] every ~20s.')
print('         RESUMABLE: re-run after any disconnect; Drive tiles ->', os.environ.get('RUNG9_CACHE'))
rc = run_logged([sys.executable, '-u', 'scripts/30_hla_ifn_inducibility.py', 'run'], RUNLOG)
print('[CELL 5]', '✓ done' if rc == 0 else f'✗ exit {rc} — re-run this cell to resume from the last cached tissue')
from IPython.display import Image, display
import json
if os.path.exists('runs/rung9_ifn/rung9_ifn.png'):
    display(Image('runs/rung9_ifn/rung9_ifn.png'))
if os.path.exists('runs/rung9_ifn/rung9_ifn_inducibility.json'):
    d = json.load(open('runs/rung9_ifn/rung9_ifn_inducibility.json'))
    print('VERDICT:', d['VERDICT'])

In [ ]:
#@title Cell 6 — bundle the WHOLE run into ONE timestamped zip + download it
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung9_ifn_', '').replace('.log', '')
bundle = f'/content/rung9_run_{stem}.zip'
paths = sorted(set(glob.glob('runs/rung9_ifn/*.png') + glob.glob('runs/rung9_ifn/*.json') + [str(RUNLOG)]))
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if os.path.exists(p):
            z.write(p, arcname=os.path.basename(p)); print('  bundled', p)
print('[CELL 6] ->', bundle)
try:
    from google.colab import files; files.download(bundle)
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What RUNG-9 decides
Resolves RUNG-8's open question. For the immune-privileged tissues (heart conduction, cardiomyocytes, neurons):
if HLA-A-low is **much lower in IFN-high cells** *and* `hk_matched` is true (depth controlled), then the
interferon program **re-arms the blocker** → the gate's safety hole shrinks in the inflamed therapeutic
context. If not, **the hole is real** even under inflammation, and the genetic NOT-gate is unsafe in heart/brain.

**Honest ceiling:** mRNA ≠ surface protein; ISG-score is a proxy for IFN exposure (resting atlas has few
IFN-high cells → that stratum can be small, n reported); depth confound controlled by housekeeping genes but
not perfectly; HLA-A gene not the A\*02 allele. Definitive answer needs IFN-stimulated / surface-protein data.